# Import Libraries

In [1]:
from transformers import AutoTokenizer
import spacy
import pandas as pd
import numpy as np
import re
import nltk
from sklearn.model_selection import train_test_split as tts
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('punkt')
nltk.download('punkt_tab')

# Load a pretrained tokenizer (BERT)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# Load SpaCy small model
nlp = spacy.load("en_core_web_sm")
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

d:\Ostad\Ostad(Deep Learning)\Deep Learning Practice\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


# Dataset

In [2]:
# Hugging Face Datasets
#!pip install --upgrade datasets huggingface_hub

from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

d:\Ostad\Ostad(Deep Learning)\Deep Learning Practice\env\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\datasets--stanfordnlp--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 370559.53 examples

In [3]:
train_data = pd.DataFrame(dataset['train'])
test_data = pd.DataFrame(dataset['test'])

In [4]:
train_data.head()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [5]:
train_data.tail()

,text,label
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1
24999,The story centers around Barry McKenzie who mu...,1


# Data Loading & Preprocessing

In [6]:
def clean_text_with_stopwords(text):
    # Convert text to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)

    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # Tokenization
    tokens = tokenizer.tokenize(text)

    # Lemmatization and removing stop words
    doc = nlp(" ".join(tokens))
    lemmas = [token.lemma_ for token in doc]
    clean_text = [word for word in lemmas if word not in stop_words]

    return " ".join(clean_text)


In [7]:
# Sample exactly 2000 rows from the original training dataset
train_sample_2000 = train_data.sample(n=2000, random_state=42)

train_df_sample, val_df_sample = tts(train_sample_2000, test_size=0.20, random_state=42)

# Sample exactly 200 rows from the original testing dataset for final evaluation
test_df_sample = test_data.sample(n=200, random_state=42)

# Print the sizes to verify the correct distribution
print(f"Total sampled from train_data: {len(train_sample_2000)}")
print(f"Training dataset size (80%): {len(train_df_sample)}")
print(f"Validation dataset size (20%): {len(val_df_sample)}")
print(f"Testing dataset size: {len(test_df_sample)}\n")

Total sampled from train_data: 2000
Training dataset size (80%): 1600
Validation dataset size (20%): 400
Testing dataset size: 200



In [8]:
train_df_sample.head()

,text,label
20257,If in the 90's you're adapting a book written ...,1
6384,I read about this movie in a magazine and I wa...,0
1288,"Ouch, what a painfully BORING Sci-Fi movie! An...",0
22091,mature intelligent and highly charged melodram...,1
20889,What a great film it is. The notion of nations...,1


In [9]:
print("Preprocessing training data... (Please wait)")
train_df_sample['clean_text_with_stopwords'] = train_df_sample['text'].apply(clean_text_with_stopwords)

print("Preprocessing validation data... (Please wait)")
val_df_sample['clean_text_with_stopwords'] = val_df_sample['text'].apply(clean_text_with_stopwords)

print("Preprocessing testing data... (Please wait)")
test_df_sample['clean_text_with_stopwords'] = test_df_sample['text'].apply(clean_text_with_stopwords)

print("\nPreprocessing completed!")
train_df_sample.head()

Preprocessing training data... (Please wait)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (837 > 512). Running this sequence through the model will result in indexing errors


Preprocessing validation data... (Please wait)
Preprocessing testing data... (Please wait)

Preprocessing completed!


,text,label,clean_text_with_stopwords
20257,If in the 90's you're adapting a book written ...,1,90 # # e adapt book write 50 # # set bloody th...
6384,I read about this movie in a magazine and I wa...,0,I read movie magazine I intrigue woman one day...
1288,"Ouch, what a painfully BORING Sci-Fi movie! An...",0,ou # # ch painfully boring sci # # fi movie # ...
22091,mature intelligent and highly charged melodram...,1,mature intelligent highly charge mel # # od # ...
20889,What a great film it is. The notion of nations...,1,great film notion nation send people fight gia...
